# 🚀 Модуль 10. Урок 2. Извлечение структурированных сущностей из неструктурированных данных (NER)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_10_llm_api/lesson_02_parsing_unstructured_data.ipynb)

## 📌 Бизнес-контекст и методология
80% корпоративных данных — неструктурированные: логи чатов саппорта, PDF-счета, отзывы на маркетплейсах.
Аналитик создает пайплайны Named Entity Recognition (NER), извлекающие атрибуты в строгую таблицу Pandas.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import re
import pandas as pd

raw_messages = [
    "Клиент ООО 'Вектор' (ИНН 7701234567) жалуется на задержку поставки на 4 дня.",
    "Компания АО 'Технопром' (ИНН 7809876543) просит увеличить лимит до 500 000 руб.",
    "ИП Смирнов (ИНН 500112233445) запросил закрытие счета."
]

extracted_records = []
for msg in raw_messages:
    inn_match = re.search(r'ИНН\s+(\d+)', msg)
    inn = inn_match.group(1) if inn_match else None
    
    org_match = re.search(r'(?:ООО|АО|ИП)\s+([^\(\)]+)', msg)
    org_name = org_match.group(0).strip() if org_match else "Unknown"
    
    extracted_records.append({
        'organization': org_name,
        'inn': inn,
        'raw_text_len': len(msg)
    })

df_entities = pd.DataFrame(extracted_records)
print("📊 Извлеченные структурированные сущности:")
print(df_entities)


📊 Извлеченные структурированные сущности:
     organization           inn  raw_text_len
0    ООО 'Вектор'    7701234567            76
1  АО 'Технопром'    7809876543            79
2      ИП Смирнов  500112233445            54


### 🎯 Практическое задание 1: Валидация полноты извлечения
Убедитесь, что для всех 3 сообщений ИНН был успешно извлечен без пропусков.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert df_entities['inn'].notna().all(), "Все ИНН должны быть извлечены"
assert len(df_entities) == 3, "Должно быть 3 записи"
print("✅ Тест успешно пройден: неструктурированные данные преобразованы в Dataframe!")


✅ Тест успешно пройден: неструктурированные данные преобразованы в Dataframe!


### 💼 Пайплайн обработки текста в 2027 году
LLM извлекает сложные семантические сущности (тональность, контекст претензии), а быстрые RegEx валидируют строгие форматы (ИНН, номера заказов, email).